### Neural alignment to behavior
- feeder rasters separated by open/closed, sorted by feeder ID and interaction duration
- feeder tuning curves warped to span ~1 sec(?) before start to ~1 sec after end, split by feeder/state
    - or whisker plot avg firing before, during, after split by feeder/state w/ dashed line for baseline FR?
- place map?
- eating raster sorted by duration
- eating tuning curve warped to span ~1 sec(?) before start to ~1 sec after end
    - or whisker plot avg firing before, during, after w/ dashed line for baseline FR?
- cache raster sorted by duration
- retrieve raster sorted by duration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.cm import get_cmap

from scipy import stats
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from skimage import measure

import sys
sys.path.append("..//neural/")
import format_waveform_data, waveform_analysis
sys.path.append("..//utils/")
from helpers import nan_interp
from load_matlab_data import loadmat_sbx

import scipy.io
import os
import mat73
from scipy.io import loadmat, savemat

In [ ]:
''' set paths '''
root_dir = "Z:/Isabel/data/"
bird_id = 'SLV132'
session_id = '250328'
locker_root = f"{root_dir}hpc_implants/{bird_id}/{bird_id}_{session_id}/"
local_root = "../data/antidromic_stim/"

# to load waveforms and probe info
ephys_id = "SLV132_250328_104521"
ks_dir = "kilosort4_blanked/"
# session_dir = f"{locker_root}/{ephys_id}/" # locker
session_dir = f"{local_root}{bird_id}/{bird_id}_{session_id}/" # local
ephys_dir = f"{session_dir}raw_ephys_output/"

# specify broken channels for this session
'''
AMB154_241119 - broken_channels = np.asarray([3, 23, 28, 29, 41, 48, 50, 61, 63])
RBY94_241129 - broken_channels = np.asarray([14, 22, 35, 42, 48])

SLV132_250303 - broken_channels = np.asarray([53, 56, 57, 63])
SLV132_250326 - np.asarray([40, 46, 53, 56, 63])
 B-07-1, B-09-1, B-11-1, B-15-3, B-17-3, B-22-2
SLV132_250328 - np.asarray([53, 56, 63])

SPP132_250523 - broken_channels = np.asarray([30, 47, 53, 59, 60, 61, 62, 63])
RBY92 - np.asarray([14, 15, 22, 32, 35, 48, 62])
PRL72 - np.asarray([30, 47, 53, 56, 59, 63])
LVN7 - np.asarray([14, 18, 20, 22, 23, 28, 29, 33, 37, 47, 48, 54, 59, 61, 63])
'''
broken_channels = np.asarray([53, 56, 63])

# to load raw keypoints, model performance measures, behavior
pred_file = f'250402_posture_2stage_face.npy'
data_dir = f"{locker_root}behavior_data/"
pred_path = f"{data_dir}{pred_file}"
annotated_seed_file = "annotatedSeeds.mat"
# annotated_seed_file = "annotatedSeeds_errors.mat"

# feeder times (minutes)
feeder_open = np.asarray([10, 55, 110])
feeder_close = np.asarray([22, 65, 117])

# arena model
arena_folder = 'Z:/Isabel/arena/v0/arena_model/'
image_file = 'arena_model-01.png'
arena_dir = 'C:/Users/ilow1/Documents/code/il_rig_control/arena_alignment/'
arena_items_file = 'arena_items_2.mat'

In [ ]:
# set save folder
hi_res = False # to save nice versions of interesting cells
save_dir = f"../figures/basic_neural_analysis/{bird_id}/HHMI2025/"
if os.path.isdir(save_dir):
    print('save directory exists')
else:
    os.mkdir(save_dir)
save_folder = f"{save_dir}/{bird_id}_{session_id}/"
if os.path.isdir(save_folder):
    print('save folder exists')
else:
    os.mkdir(save_folder)
save_subfolder = f"{save_folder}summary_figs/"
if os.path.isdir(save_subfolder):
    print('save sub-folder exists')
else:
    os.mkdir(save_subfolder)